In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [7]:
rng = np.random.default_rng(0)
OUT = "."
plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "figure.dpi": 150, "savefig.bbox": "tight"})
ALPHA = 0.10

# ---------- Regresión: datos y modelos por cuantiles (pinball loss) ----------
X = rng.uniform(0, 10, size=(3000, 1))
y = np.sin(X[:, 0]) + rng.normal(0, 0.2 + 0.15 * X[:, 0])
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=1)

models = {}
for tau in (0.05, 0.50, 0.95):
    m = HistGradientBoostingRegressor(loss="quantile", quantile=tau,
                                      max_iter=200, random_state=0)
    m.fit(X_tr, y_tr)
    models[tau] = m

def q_pred(X_, tau):
    return models[tau].predict(X_)

q05_te, q50_te, q95_te = q_pred(X_te, 0.05), q_pred(X_te, 0.50), q_pred(X_te, 0.95)
q05_cal, q95_cal = q_pred(X_cal, 0.05), q_pred(X_cal, 0.95)
cov_raw = np.mean((y_te >= q05_te) & (y_te <= q95_te))
E = np.maximum(q05_cal - y_cal, y_cal - q95_cal)
n_cal = len(E)
k = int(np.ceil((n_cal + 1) * (1 - ALPHA)))
Q = np.quantile(E, k / n_cal) if k <= n_cal else np.inf
lo_cqr, hi_cqr = q05_te - Q, q95_te + Q
cov_cqr = np.mean((y_te >= lo_cqr) & (y_te <= hi_cqr))
print(f"regresión: n_cal={n_cal} Q={Q:.3f} cobertura cruda={cov_raw:.3f} "
      f"cobertura CQR={cov_cqr:.3f} ancho medio CQR={(hi_cqr-lo_cqr).mean():.3f}")

xs = np.linspace(0, 10, 400).reshape(-1, 1)
q05_x, q50_x, q95_x = q_pred(xs, 0.05), q_pred(xs, 0.50), q_pred(xs, 0.95)
# Suavizado SOLO para visualización (el modelo de árboles da curvas
# escalonadas); las métricas usan las predicciones sin suavizar.
def _smooth(v, w=21):
    pad = np.pad(v, w // 2, mode="edge")
    return np.convolve(pad, np.ones(w) / w, mode="valid")
q05_x, q50_x, q95_x = _smooth(q05_x), _smooth(q50_x), _smooth(q95_x)

# Fig 1: punto vs intervalo (modelo entrenado)
fig, ax = plt.subplots(figsize=(8, 4.6))
inside = (y_te >= q05_te) & (y_te <= q95_te)
ax.fill_between(xs[:, 0], q05_x, q95_x, color="#2f6fb2", alpha=0.25,
                label="Intervalo")
ax.plot(xs[:, 0], q50_x, color="#2f6fb2", lw=2, label="Mediana $q_{0.50}$ (punto)")
ax.scatter(X_te[inside, 0], y_te[inside], s=14, color="#222222", alpha=0.6,
           label="y dentro del intervalo")
ax.scatter(X_te[~inside, 0], y_te[~inside], s=28, color="#c0392b",
           label="y fuera del intervalo")
ax.set_xlabel("x"); ax.set_ylabel("y")
#ax.set_title("Un punto no es una garantía: el intervalo sí dice dónde puede caer y")
ax.legend(loc="upper left", fontsize=9)
fig.savefig(f"{OUT}/fig1_intervalo_vs_punto.png");
plt.close(fig)


regresión: n_cal=600 Q=0.288 cobertura cruda=0.803 cobertura CQR=0.905 ancho medio CQR=3.259
